# Test Snowflake UDF: EXTRACT_FEATURES_UDF
Notebook prêt a executer dans Snowflake pour tester `EXTRACT_FEATURES_UDF` sur un fichier audio depuis un stage.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

# Parametres
PROCESSED_STAGE_PATH = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS/asthma/"
CLASS_NAME = "Asthma"
AUDIO_FILE = None  # ex: 'P13WheezingIE_61.wav' ; None = auto-pick premier .wav
OUTPUT_STAGE = "@M2_ISD_EQUIPE_1_DB.TEST.STG_RESPIRATORY_FEATURES"  # utilise seulement si SAVE_TO_STAGE=True
SAVE_TO_STAGE = False

print("Session OK")
print("Stage :", PROCESSED_STAGE_PATH)
print("Class :", CLASS_NAME)

In [ ]:
# Pre-check: contexte SQL + existence de la UDF
ctx = session.sql("""
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH
""").to_pandas()
print(ctx.to_string(index=False))

udf_check = session.sql("""
SHOW USER FUNCTIONS LIKE 'EXTRACT_FEATURES_UDF' IN SCHEMA M2_ISD_EQUIPE_1_DB.PROCESSED
""").to_pandas()

print(f"UDFs trouvees: {len(udf_check)}")
print("Colonnes retournees:", list(udf_check.columns))

if not udf_check.empty:
    lower_map = {str(c).lower(): c for c in udf_check.columns}
    wanted = ["name", "schema_name", "arguments", "language"]
    present = [lower_map[w] for w in wanted if w in lower_map]
    display(udf_check[present] if present else udf_check)
else:
    print("EXTRACT_FEATURES_UDF non trouvee dans M2_ISD_EQUIPE_1_DB.PROCESSED")

In [ ]:
# Liste les fichiers du stage et choisit un .wav
ls_df = session.sql(f"LIST {PROCESSED_STAGE_PATH}").to_pandas()
if ls_df.empty:
    raise ValueError(f"Aucun fichier trouve dans {PROCESSED_STAGE_PATH}")

ls_df["FILE_NAME"] = ls_df["name"].str.split('/').str[-1]
audio_candidates = ls_df[ls_df["FILE_NAME"].str.lower().str.endswith(".wav", na=False)]
if audio_candidates.empty:
    raise ValueError("Aucun fichier .wav trouve dans le stage")

if AUDIO_FILE is None:
    AUDIO_FILE = audio_candidates.iloc[0]["FILE_NAME"]

print("Fichier teste :", AUDIO_FILE)
display(audio_candidates[["FILE_NAME", "size", "last_modified"]].head(20))

In [ ]:
# Appel de EXTRACT_FEATURES_UDF avec diagnostic
def sql_escape(s: str) -> str:
    return s.replace("'", "''")

audio_file_sql = sql_escape(AUDIO_FILE)
stage_sql = sql_escape(PROCESSED_STAGE_PATH)
class_sql = sql_escape(CLASS_NAME)
out_stage_sql = sql_escape(OUTPUT_STAGE)

# Signature UDF:
# EXTRACT_FEATURES_UDF(processed_audio_path, stage_name, file_name, class_name, save_to_stage, output_stage)
query = f"""
SELECT M2_ISD_EQUIPE_1_DB.PROCESSED.EXTRACT_FEATURES_UDF(
  '{audio_file_sql}',
  '{stage_sql}',
  '{audio_file_sql}',
  '{class_sql}',
  {str(SAVE_TO_STAGE).upper()},
  {'NULL' if not SAVE_TO_STAGE else f"'{out_stage_sql}'"}
) AS RESULT
"""

print("Executing SQL:
", query)

try:
    rows = session.sql(query).collect()
    if not rows:
        raise RuntimeError("La requete a retourne 0 ligne")

    result = rows[0]["RESULT"]
    print("Result STATUS:", result.get("STATUS"))
    if result.get("STATUS") == "ERROR":
        print("Result ERROR:", result.get("ERROR"))
    else:
        print("Feature types:", list(result.get("FEATURES", {}).keys()))
    result
except Exception as e:
    print("Echec appel UDF")
    print("Type:", type(e).__name__)
    print("Message:", str(e))
    print("\nChecks rapides:")
    print("1) UDF deployee dans M2_ISD_EQUIPE_1_DB.PROCESSED")
    print("2) Le fichier existe dans le stage")
    print("3) libroza.zip dispo dans @M2_ISD_EQUIPE_1_DB.PUBLIC.STG_LIBRARIES/libs/")
    raise

In [ ]:
# Affichage resumee des features si SUCCESS
if 'result' in globals() and isinstance(result, dict) and result.get('STATUS') == 'SUCCESS':
    features = result.get('FEATURES', {})
    rows = []
    for k, v in features.items():
        rows.append({
            'FEATURE_TYPE': k,
            'SHAPE': v.get('shape'),
            'DTYPE': v.get('dtype'),
            'N_FRAMES': v.get('n_frames'),
            'N_COEFFICIENTS': v.get('n_coefficients'),
        })
    display(pd.DataFrame(rows))
else:
    print('Pas de resultat SUCCESS a resumer.')